In [ ]:
# Diffusion Policy (clone gchenfc/diffusion_policy then `pip install -e .`)
from diffusion_policy.common.replay_buffer import ReplayBuffer

# GML
import json
from pathlib import Path

import load_gml
import numpy as np
import tqdm.notebook as tqdm
from load_gml import Drawing

%load_ext autoreload
%autoreload 2

In [ ]:
MODE = 'by_stroke'
# MODE = 'by_drawing'
SCALE_BEHAVIOR = 'GML_SCREEN'  # default
SCALE_BEHAVIOR = 'PRESERVE_ASPECT_CENTERED'
# SCALE_BEHAVIOR = 'PRESERVE_ASPECT'
# SCALE_BEHAVIOR = 'NONE'

In [ ]:
# Quickly visualize the structure of gml files
drawing = load_gml.get_from_blackbook(30083, verbosity=3, scale_behavior=SCALE_BEHAVIOR)
print(drawing.strokes[0].shape)
with np.printoptions(precision=3, suppress=True):
    print(drawing.strokes[0][:10])
# t, x, y, ?z

In [ ]:
def compute_state(stroke):
    return stroke[:, 1:3].astype(np.float32)
def compute_action(stroke):
    return np.diff(stroke[:, 1:3], axis=0, append=stroke[-1:, 1:3]).astype(np.float32)
def compute_obs(stroke):
    return None

In [ ]:
# sorted_fnames = reversed(sorted(FOLDER.glob('*.json'), key=lambda x: int(x.with_suffix('').name)))
# katsu_fnames = load_gml.filter_by_application(sorted_fnames)
# len(list(katsu_fnames))
# 46391

In [ ]:
def Drawing(item, **kwargs):
    try:
        return load_gml.Drawing(item, **kwargs)
    except Exception:
        return None

In [ ]:
FOLDER = Path('data/gml')
outprefix = f'data/gml_{MODE}' + (f'_{SCALE_BEHAVIOR}' if SCALE_BEHAVIOR != 'GML_SCREEN' else '')
sorted_fnames = reversed(sorted(FOLDER.glob('*.json'), key=lambda x: int(x.with_suffix('').name)))
katsu_fnames = load_gml.filter_by_application(sorted_fnames)
if SCALE_BEHAVIOR == 'GML_SCREEN':
    drawings = map(Drawing, katsu_fnames)
else:
    drawings = map(lambda x: Drawing(x, scale_behavior=SCALE_BEHAVIOR), katsu_fnames)

dataset = ReplayBuffer.create_empty_zarr()
for i, drawing in tqdm.tqdm(enumerate(drawings), total=46391):
    if drawing is None:
        continue
    if MODE == 'by_stroke':
        for stroke in drawing.strokes:
            dataset.add_episode({
                'state': compute_state(stroke),
                'action': compute_action(stroke),
                # 'obs': compute_obs(stroke),
            })
    elif MODE == 'by_drawing':
        dataset.add_episode({
            'state': np.concatenate([compute_state(stroke) for stroke in drawing.strokes], axis=0),
            'action': np.concatenate([compute_action(stroke) for stroke in drawing.strokes], axis=0),
            # 'obs': compute_obs(stroke),
        })
    else:
        raise ValueError(f'Unknown mode: {MODE}')
    if i % 3000 == 0:
        dataset.save_to_path(f'{outprefix}_{i:06d}.zarr')

In [ ]:
dataset.save_to_path('data/gml.zarr')

In [ ]:
import matplotlib.pyplot as plt
e = 0
for i in range(100):
    s, e = e, dataset.episode_ends[i]
    plt.plot(dataset.data.state[s:e, 0], dataset.data.state[s:e, 1])
    # print(dataset.episode_starts[i], dataset.episode_ends[i], dataset.episode_lengths[i])